# BirdCLEF+ 2026 — Phase 3 Submission (Perch v2 + Head)

Loads Perch ONNX from `tuckerarrants/perch-v2-no-dft-onnx` dataset, our trained head
from `harishteens/birdclef-2026-baseline-ckpt`, runs inference on the hidden test set.

Internet is disabled at scored runtime — both models must be pre-attached as datasets.


In [ ]:
import subprocess, sys, os
from pathlib import Path

KI = Path("/kaggle/input")

def find_one(pattern):
    matches = sorted(KI.rglob(pattern))
    assert matches, f"No file matched {pattern} under /kaggle/input — is the dataset attached?"
    return matches[0]

# Install onnxruntime from the bundled wheel (no internet at scored runtime)
try:
    import onnxruntime as ort
    print(f"onnxruntime already available: {ort.__version__}")
except ImportError:
    ort_whl = find_one("onnxruntime-*.whl")
    print(f"installing from {ort_whl}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps", str(ort_whl)])
    import onnxruntime as ort
    print(f"onnxruntime installed: {ort.__version__}")

import time
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import torch
import torch.nn as nn
from scipy.ndimage import convolve1d

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "  torch:", torch.__version__)


In [ ]:
# Find the Perch ONNX, regardless of which path Kaggle mounted it at
PERCH_PATH = find_one("perch_v2_no_dft.onnx")
print("Perch ONNX:", PERCH_PATH)

# Find our checkpoint
CKPT_DSET = next(KI.rglob("birdclef-2026-baseline-ckpt"))
print("Checkpoint dir:", CKPT_DSET)
for cand in ["model_v5.pt", "model_v2.pt", "model.pt"]:
    CKPT_PATH = CKPT_DSET / cand
    if CKPT_PATH.exists():
        break
else:
    raise FileNotFoundError(f"No checkpoint in {CKPT_DSET}")
print("Using checkpoint:", CKPT_PATH)

# Competition data + paths
COMP_DIR  = next(KI.rglob("birdclef-2026"))
# birdclef-2026 might match our ckpt dir name too — pick the one with the right children
if not (COMP_DIR / "test_soundscapes").is_dir():
    for cand in KI.rglob("birdclef-2026"):
        if (cand / "test_soundscapes").is_dir():
            COMP_DIR = cand
            break
TEST_DIR  = COMP_DIR / "test_soundscapes"
TRAIN_DIR = COMP_DIR / "train_soundscapes"
OUT_PATH  = Path("/kaggle/working/submission.csv")
print("Comp dir:", COMP_DIR)


In [ ]:
# ---- Load Perch --------------------------------------------------------
sess = ort.InferenceSession(str(PERCH_PATH), providers=["CPUExecutionProvider"])
INPUT_NAME    = sess.get_inputs()[0].name
EMBED_OUT_IDX = next(i for i, o in enumerate(sess.get_outputs()) if o.name == "embedding")
print(f"Perch loaded. Embedding output idx={EMBED_OUT_IDX}")


In [ ]:
# ---- Load all 5 seed heads for ensemble --------------------------------
import torch.nn as nn

class PerchHead(nn.Module):
    def __init__(self, embed_dim=1536, hidden_dim=512, num_classes=234, dropout=0.3):
        super().__init__()
        self.norm  = nn.LayerNorm(embed_dim)
        self.drop1 = nn.Dropout(0.2)
        self.fc1   = nn.Linear(embed_dim, hidden_dim)
        self.act   = nn.ReLU(inplace=True)
        self.drop2 = nn.Dropout(dropout)
        self.fc2   = nn.Linear(hidden_dim, num_classes)
    def forward(self, x):
        x = self.norm(x)
        x = self.drop1(x)
        x = self.act(self.fc1(x))
        x = self.drop2(x)
        return self.fc2(x)


# Find all seed checkpoints in the dataset dir
seed_ckpts = sorted(CKPT_DSET.glob("model_v5_seed*.pt"))
if not seed_ckpts:
    # Fall back to single-head checkpoints if no seeds present
    seed_ckpts = [CKPT_PATH]

print(f"Loading {len(seed_ckpts)} head checkpoint(s):")
for p in seed_ckpts:
    print(f"  {p.name}")

heads = []
species, label_to_idx, NUM_CLASSES, EMBED_DIM = None, None, None, None
for ckpt_path in seed_ckpts:
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    if species is None:
        species       = ckpt["species"]
        label_to_idx  = ckpt["label_to_idx"]
        NUM_CLASSES   = ckpt["num_classes"]
        EMBED_DIM     = ckpt["embed_dim"]
    h = PerchHead(EMBED_DIM, **ckpt["head_config"], num_classes=NUM_CLASSES).to(DEVICE)
    h.load_state_dict(ckpt["state_dict"])
    h.eval()
    heads.append(h)

print(f"Heads loaded: {len(heads)}  classes={NUM_CLASSES}")


In [ ]:
# ---- Audio + windowing -------------------------------------------------
SR          = 32000
CLIP_SEC    = 5
N_SAMPLES   = SR * CLIP_SEC
WINDOW_SEC  = CLIP_SEC
N_WINDOWS   = 60 // WINDOW_SEC
GAUSSIAN_KERNEL = np.array([0.1, 0.2, 0.4, 0.2, 0.1])


def load_audio(path):
    wav, sr = sf.read(str(path), dtype="float32", always_2d=False)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if sr != SR:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)
    return wav.astype(np.float32)


def file_to_chunks(path):
    wav    = load_audio(path)
    target = N_WINDOWS * N_SAMPLES
    if len(wav) < target:
        wav = np.pad(wav, (0, target - len(wav)))
    else:
        wav = wav[:target]
    return wav.reshape(N_WINDOWS, N_SAMPLES).astype(np.float32)


def smooth_windows(logits):
    return convolve1d(logits, GAUSSIAN_KERNEL, axis=0, mode="nearest")


@torch.no_grad()
def predict_file(path):
    chunks = file_to_chunks(path)
    embeddings = sess.run(None, {INPUT_NAME: chunks})[EMBED_OUT_IDX]
    embeddings = torch.from_numpy(embeddings).to(DEVICE)
    logits     = sum(h(embeddings) for h in heads).cpu().numpy() / len(heads)
    logits     = smooth_windows(logits)
    return 1.0 / (1.0 + np.exp(-logits))


In [ ]:
# ---- Find test files (fall back to train_soundscapes for the preview run) ----
test_files = sorted(TEST_DIR.glob("*.ogg")) if TEST_DIR.is_dir() else []
if not test_files:
    test_files = sorted(TRAIN_DIR.glob("*.ogg"))[:5]
    print(f"[fallback] using {len(test_files)} train soundscapes")
else:
    print(f"Found {len(test_files)} test files")


In [ ]:
# ---- Inference loop ----------------------------------------------------
all_rows, all_probs = [], []
t0 = time.time()
for i, f in enumerate(test_files):
    basename = f.stem
    probs    = predict_file(f)
    end_secs = np.arange(1, N_WINDOWS + 1) * WINDOW_SEC
    for k in range(N_WINDOWS):
        all_rows.append(f"{basename}_{end_secs[k]}")
        all_probs.append(probs[k])
    if (i + 1) % 25 == 0 or i == 0 or i == len(test_files) - 1:
        dt = time.time() - t0
        rate = (i + 1) / max(dt, 1e-9)
        print(f"  [{i+1:4d}/{len(test_files)}] {dt:.1f}s  {rate:.2f} files/s")

all_probs = np.stack(all_probs)
print(f"Inference done: {len(all_rows)} rows in {time.time()-t0:.1f}s")


In [ ]:
# ---- Build submission --------------------------------------------------
sample_sub = pd.read_csv(COMP_DIR / "sample_submission.csv")
all_species_in_order = [c for c in sample_sub.columns if c != "row_id"]

pred_df = pd.DataFrame(all_probs, columns=species)
pred_df.insert(0, "row_id", all_rows)
sub = pred_df.set_index("row_id").reindex(columns=all_species_in_order)
sub = sub.fillna(1.0 / len(all_species_in_order)).clip(0.0, 1.0).reset_index()

assert list(sub.columns) == list(sample_sub.columns)
assert sub["row_id"].is_unique
assert not sub.isna().any().any()

sub.to_csv(OUT_PATH, index=False)
print(f"Wrote {OUT_PATH}  shape={sub.shape}  size={OUT_PATH.stat().st_size/1024:.1f} KB")
sub.head(3)
